# ARIA v3.0 — Typhoon Fung-wong Dynamic Risk Integration

This notebook integrates:

- **W3** river-distance shelter exposure
- **W4** terrain slope / elevation risk
- **Week 5** live-or-simulated CWA rainfall data

Final deliverable:
- **Interactive map**: `ARIA_v3_Fungwong.html`
- **Format**: `.ipynb`

The workflow is **infrastructure first**:
1. Read configuration from `.env`
2. Support `LIVE` and `SIMULATION` modes
3. Normalize CWA JSON into one schema
4. Fall back to the latest local snapshot when API fetch fails
5. Run downstream spatial analysis **without changing later logic**

## Captain's Log 1 — Environment, paths, thresholds

This notebook is **path-cleaned** for grading and portability.

Path resolution priority:
1. Explicit paths from `.env`
2. Relative paths from the notebook / `.env` directory
3. Auto-discovery by filename pattern under the project roots

This avoids machine-specific hard-coded paths such as `Downloads/...` inside the notebook code.


In [15]:
import os
import json
import warnings
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
import rioxarray as rxr

from shapely.geometry import Point
from rasterstats import zonal_stats
from folium.plugins import HeatMap
from folium import FeatureGroup, LayerControl
from IPython.display import display

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    load_dotenv = None
    find_dotenv = None

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# Path / environment bootstrap
# Goal: no hard-coded machine-specific paths in the notebook itself.
# Resolution priority:
#   1) explicit path from .env
#   2) relative path from .env directory / project directory
#   3) auto-discovery by filename pattern under project roots
# -------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd().resolve()

def _clean_env_path(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    return str(value).replace("遙測 /", "遙測/").strip().strip('"').strip("'")

def _find_env_file(start_dir: Path) -> Optional[Path]:
    candidates = []

    if find_dotenv is not None:
        found = find_dotenv(filename=".env", usecwd=True)
        if found:
            candidates.append(Path(found))

    for base in [start_dir, *start_dir.parents]:
        candidates.append(base / ".env")
        candidates.append(base / "hw5" / ".env")

    for p in candidates:
        if p.exists():
            return p.resolve()
    return None

ENV_FILE = _find_env_file(NOTEBOOK_DIR)
if load_dotenv is not None and ENV_FILE is not None:
    load_dotenv(ENV_FILE, override=True)
elif load_dotenv is not None:
    load_dotenv(override=True)

PROJECT_DIR = ENV_FILE.parent.resolve() if ENV_FILE is not None else NOTEBOOK_DIR
ROOT_DIR_RAW = _clean_env_path(os.getenv("ROOT_DIR"))
if ROOT_DIR_RAW:
    ROOT_DIR = Path(ROOT_DIR_RAW).expanduser()
    if not ROOT_DIR.is_absolute():
        ROOT_DIR = (PROJECT_DIR / ROOT_DIR).resolve()
    else:
        ROOT_DIR = ROOT_DIR.resolve()
else:
    ROOT_DIR = PROJECT_DIR

def _search_roots() -> list[Path]:
    roots = [PROJECT_DIR, ROOT_DIR]
    if PROJECT_DIR.parent not in roots:
        roots.append(PROJECT_DIR.parent)
    return [p.resolve() for p in roots if p.exists()]

SEARCH_ROOTS = _search_roots()

def _resolve_candidate_path(raw: Optional[str]) -> Optional[Path]:
    raw = _clean_env_path(raw)
    if not raw:
        return None

    p = Path(raw).expanduser()
    if p.is_absolute():
        return p.resolve()

    # Relative paths are interpreted relative to:
    #   1) .env file directory
    #   2) ROOT_DIR
    #   3) current notebook directory
    for base in [PROJECT_DIR, ROOT_DIR, NOTEBOOK_DIR]:
        candidate = (base / p).resolve()
        if candidate.exists():
            return candidate

    # Return the first interpreted location even if it doesn't exist yet,
    # so the debug print shows the intended path.
    return (PROJECT_DIR / p).resolve()

def _discover_by_patterns(patterns: Iterable[str], kind: str) -> Optional[Path]:
    matches: list[Path] = []
    seen = set()
    for root in SEARCH_ROOTS:
        for pattern in patterns:
            for p in root.rglob(pattern):
                if p.is_file() and p.resolve() not in seen:
                    matches.append(p.resolve())
                    seen.add(p.resolve())

    if not matches:
        return None

    # Prefer paths closest to project/root directories and newest among ties.
    def score(p: Path):
        name = p.name.lower()
        preferred = 0
        if "town_moi" in name or "fungwong" in name or "dem" in name or "避難" in name:
            preferred -= 10
        depth = len(p.parts)
        return (preferred, depth, -p.stat().st_mtime)

    matches = sorted(matches, key=score)
    chosen = matches[0]
    print(f"Auto-discovered {kind}: {chosen}")
    return chosen

def resolve_data_path(
    env_name: str,
    *,
    default_relative: Optional[str] = None,
    discovery_patterns: Optional[list[str]] = None,
    required: bool = True,
) -> Optional[Path]:
    # 1) explicit path from .env
    env_value = os.getenv(env_name)
    candidate = _resolve_candidate_path(env_value)
    if candidate is not None and candidate.exists():
        return candidate

    # 2) relative default from project / root
    if default_relative:
        for base in [PROJECT_DIR, ROOT_DIR, NOTEBOOK_DIR]:
            p = (base / default_relative).resolve()
            if p.exists():
                return p

    # 3) discovery fallback
    if discovery_patterns:
        discovered = _discover_by_patterns(discovery_patterns, env_name)
        if discovered is not None:
            return discovered

    # 4) return best-effort candidate or fail
    if candidate is not None:
        if required:
            raise FileNotFoundError(
                f"{env_name} points to a missing path: {candidate}\n"
                f"Please fix it in {ENV_FILE or '.env'}."
            )
        return candidate

    if required:
        raise FileNotFoundError(
            f"Could not resolve required path for {env_name}.\n"
            f"Search roots: {[str(p) for p in SEARCH_ROOTS]}\n"
            f"Please set {env_name} in {ENV_FILE or '.env'}."
        )
    return None

# ---------- App mode ----------
APP_MODE = os.getenv("APP_MODE", "SIMULATION").strip().upper()
CWA_API_KEY = os.getenv("CWA_API_KEY", "").strip()
TARGET_COUNTY = os.getenv("TARGET_COUNTY", "花蓮縣").strip()

# ---------- Thresholds ----------
BUFFER_RAIN_M = int(float(os.getenv("BUFFER_RAIN_M", 5000)))
BUFFER_SHELTER_M = int(float(os.getenv("BUFFER_SHELTER_M", 500)))
SLOPE_THRESHOLD = float(os.getenv("SLOPE_THRESHOLD", 30))
ELEVATION_LOW = float(os.getenv("ELEVATION_LOW", 50))
RAIN_WARNING_MM = float(os.getenv("RAIN_WARNING_MM", 40))
RAIN_CRITICAL_MM = float(os.getenv("RAIN_CRITICAL_MM", 80))

# ---------- Local data paths ----------
township_path = resolve_data_path(
    "TOWNSHIP_PATH",
    discovery_patterns=["TOWN_MOI_*.shp", "**/TOWN_MOI_*.shp"],
)

shelter_csv_path = resolve_data_path(
    "SHELTER_CSV_PATH",
    discovery_patterns=["避難收容處所*.csv", "*避難*收容*.csv", "*shelter*.csv"],
)

dem_path = resolve_data_path(
    "DEM_PATH",
    discovery_patterns=["dem_20m_hualien.tif", "*dem*hualien*.tif", "*dem*.tif"],
)

SIMULATION_DATA = resolve_data_path(
    "SIMULATION_DATA",
    discovery_patterns=["fungwong_*.json", "*fungwong*.json", "*rain*.json"],
)

OUTPUT_DIR = _resolve_candidate_path(os.getenv("OUTPUT_DIR")) or (PROJECT_DIR / "output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Water Resources Agency river polygons
river_zip_url = "https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP"
river_extract_dir = OUTPUT_DIR / "riverpoly"

# Final outputs
html_output_path = OUTPUT_DIR / "ARIA_v3_Fungwong.html"
csv_output_path = OUTPUT_DIR / "ARIA_v3_shelters.csv"
geojson_output_path = OUTPUT_DIR / "ARIA_v3_shelters.geojson"

config_df = pd.DataFrame([
    ["ENV_FILE", str(ENV_FILE) if ENV_FILE else "(not found; using current environment only)", True],
    ["PROJECT_DIR", str(PROJECT_DIR), PROJECT_DIR.exists()],
    ["ROOT_DIR", str(ROOT_DIR), ROOT_DIR.exists()],
    ["TOWNSHIP_PATH", str(township_path), Path(township_path).exists()],
    ["SHELTER_CSV_PATH", str(shelter_csv_path), Path(shelter_csv_path).exists()],
    ["DEM_PATH", str(dem_path), Path(dem_path).exists()],
    ["SIMULATION_DATA", str(SIMULATION_DATA), Path(SIMULATION_DATA).exists()],
    ["OUTPUT_DIR", str(OUTPUT_DIR), OUTPUT_DIR.exists()],
], columns=["key", "value", "exists"])

print("=== ARIA v3.0 configuration ===")
print("APP_MODE:", APP_MODE)
print("TARGET_COUNTY:", TARGET_COUNTY)
print("BUFFER_RAIN_M:", BUFFER_RAIN_M)
print("BUFFER_SHELTER_M:", BUFFER_SHELTER_M)
print("SLOPE_THRESHOLD:", SLOPE_THRESHOLD)
print("RAIN_WARNING_MM:", RAIN_WARNING_MM)
print("RAIN_CRITICAL_MM:", RAIN_CRITICAL_MM)
display(config_df)


=== ARIA v3.0 configuration ===
APP_MODE: SIMULATION
TARGET_COUNTY: 花蓮縣
BUFFER_RAIN_M: 5000
BUFFER_SHELTER_M: 500
SLOPE_THRESHOLD: 30.0
RAIN_WARNING_MM: 40.0
RAIN_CRITICAL_MM: 80.0


,key,value,exists
0,ENV_FILE,C:\Users\admin\Desktop\遙測\hw5\.env,True
1,PROJECT_DIR,C:\Users\admin\Desktop\遙測\hw5,True
2,ROOT_DIR,C:\Users\admin\Desktop\遙測\hw5,True
3,TOWNSHIP_PATH,C:\Users\admin\Desktop\遙測\hw5\TOWN_MOI_1140318...,True
4,SHELTER_CSV_PATH,C:\Users\admin\Desktop\遙測\hw5\避難收容處所點位檔案v9.csv,True
5,DEM_PATH,C:\Users\admin\Desktop\遙測\hw5\dem_20m_hualien.tif,True
6,SIMULATION_DATA,C:\Users\admin\Desktop\遙測\hw5\fungwong_202511....,True
7,OUTPUT_DIR,C:\Users\admin\Desktop\遙測\hw5\output,True


## Captain's Log 2 — Core helper functions

Create helper functions for:

- fetching the CWA API
- locating the latest local rainfall snapshot
- normalizing LIVE API JSON and SIMULATION JSON into one schema
- selecting **WGS84** coordinates correctly from `GeoInfo.Coordinates`
- filtering out `-998` no-data values **before** GeoDataFrame construction
- dynamic risk grading and map symbology

This is the abstraction layer that keeps downstream analysis unchanged across modes.


In [16]:

def fetch_cwa_api(api_key: str):
    """
    Fetch real-time rainfall data from CWA.
    Returns parsed JSON on success, otherwise None.
    """
    if not api_key:
        print("LIVE mode requested, but CWA_API_KEY is empty.")
        return None

    endpoint = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    params = {
        "Authorization": api_key,
        "format": "JSON",
    }

    try:
        resp = requests.get(endpoint, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        print("CWA API fetch succeeded.")
        return data
    except Exception as e:
        print(f"CWA API fetch failed: {e}")
        return None


def get_latest_local_snapshot(preferred_path: str | os.PathLike | None = None):
    """
    Return the preferred local snapshot if it exists; otherwise search under ROOT_DIR / week5.
    """
    candidates = []

    if preferred_path:
        p = Path(str(preferred_path))
        if p.exists():
            return p
        candidates.append(p)

    for p in WEEK5_DIR.rglob("*.json"):
        if "fungwong" in p.name.lower() or "rain" in p.name.lower():
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError("No local JSON snapshot found under week5 directory.")

    candidates = [p for p in candidates if p.exists()]
    latest = max(candidates, key=lambda x: x.stat().st_mtime)
    print(f"Using local snapshot: {latest}")
    return latest


def _to_float_or_nan(value):
    """
    Convert CWA values to float.
    Sentinel values such as -998 become np.nan immediately.
    """
    if value is None:
        return np.nan
    if isinstance(value, str):
        value = value.strip()
        if value == "":
            return np.nan
    try:
        out = float(value)
    except Exception:
        return np.nan

    if out <= -998:
        return np.nan
    return out


def _coord_lon_lat(coord_item):
    """
    Extract lon/lat from one coordinate dict.
    """
    if not isinstance(coord_item, dict):
        return np.nan, np.nan

    lon = coord_item.get("StationLongitude", coord_item.get("Longitude"))
    lat = coord_item.get("StationLatitude", coord_item.get("Latitude"))
    return _to_float_or_nan(lon), _to_float_or_nan(lat)


def _extract_lon_lat(coord_block):
    """
    Handle the two known coordinate cases safely.

    LIVE API:
    - GeoInfo.Coordinates[0] = TWD67
    - GeoInfo.Coordinates[1] = WGS84

    CoLife simulation snapshot:
    - only one coordinate dict, already WGS84

    Rule:
    - Prefer the second coordinate dict when two are present
    - Otherwise use the only coordinate dict
    - Fall back to a search for a plausible lon/lat pair
    """
    if isinstance(coord_block, dict):
        return _coord_lon_lat(coord_block)

    if isinstance(coord_block, list):
        if len(coord_block) >= 2:
            lon, lat = _coord_lon_lat(coord_block[1])
            if not np.isnan(lon) and not np.isnan(lat):
                return lon, lat

        if len(coord_block) == 1:
            lon, lat = _coord_lon_lat(coord_block[0])
            if not np.isnan(lon) and not np.isnan(lat):
                return lon, lat

        for item in coord_block:
            lon, lat = _coord_lon_lat(item)
            if not np.isnan(lon) and not np.isnan(lat):
                return lon, lat

    return np.nan, np.nan


def normalize_cwa_json(data):
    """
    Normalize CWA rainfall JSON from either LIVE API or SIMULATION snapshot.

    Output columns:
    - station_id
    - station_name
    - obs_time
    - lon
    - lat
    - rain_10min
    - rain_1hr
    - rain_3hr
    - rain_6hr
    - rain_12hr
    - rain_24hr

    Important:
    - this function must always output WGS84 lon/lat
    - -998 and equivalent no-data values are converted to NaN
    - rows with unusable hourly rainfall are removed before creating the GeoDataFrame
    """
    if data is None:
        raise ValueError("normalize_cwa_json received None")

    station_list = []
    if isinstance(data, dict):
        station_list = (
            data.get("records", {}).get("Station", [])
            or data.get("cwaopendata", {}).get("location", [])
            or []
        )

    if not isinstance(station_list, list):
        raise ValueError("Unable to find records.Station[] in rainfall JSON")

    rows = []
    for s in station_list:
        if not isinstance(s, dict):
            continue

        geo_info = s.get("GeoInfo", {})
        coord_block = geo_info.get("Coordinates", s.get("Coordinates", []))
        lon, lat = _extract_lon_lat(coord_block)

        rainfall = s.get("RainfallElement", {})

        rows.append({
            "station_id": s.get("StationId", s.get("stationId", "")),
            "station_name": s.get("StationName", s.get("stationName", "")),
            "obs_time": (
                s.get("ObsTime", {}).get("DateTime")
                if isinstance(s.get("ObsTime", {}), dict)
                else s.get("ObsTime")
            ),
            "lon": lon,
            "lat": lat,
            "rain_10min": _to_float_or_nan(rainfall.get("Past10Min", {}).get("Precipitation") if isinstance(rainfall.get("Past10Min"), dict) else rainfall.get("Past10Min")),
            "rain_1hr": _to_float_or_nan(rainfall.get("Past1hr", {}).get("Precipitation") if isinstance(rainfall.get("Past1hr"), dict) else rainfall.get("Past1hr")),
            "rain_3hr": _to_float_or_nan(rainfall.get("Past3hr", {}).get("Precipitation") if isinstance(rainfall.get("Past3hr"), dict) else rainfall.get("Past3hr")),
            "rain_6hr": _to_float_or_nan(rainfall.get("Past6hr", {}).get("Precipitation") if isinstance(rainfall.get("Past6hr"), dict) else rainfall.get("Past6hr")),
            "rain_12hr": _to_float_or_nan(rainfall.get("Past12hr", {}).get("Precipitation") if isinstance(rainfall.get("Past12hr"), dict) else rainfall.get("Past12hr")),
            "rain_24hr": _to_float_or_nan(rainfall.get("Past24hr", {}).get("Precipitation") if isinstance(rainfall.get("Past24hr"), dict) else rainfall.get("Past24hr")),
        })

    df = pd.DataFrame(rows)

    # Pre-GDF filtering: keep only valid WGS84 coordinates and usable 1-hour rainfall
    df = df.dropna(subset=["lon", "lat", "rain_1hr"]).copy()

    # Optional range sanity check for WGS84 Taiwan vicinity
    df = df[df["lon"].between(118, 123.5) & df["lat"].between(21, 26.5)].copy()

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    return gdf


def minmax_scale(series):
    s = series.astype(float)
    smin, smax = s.min(), s.max()
    if pd.isna(smin) or pd.isna(smax) or smax == smin:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - smin) / (smax - smin)


def rain_color(rain_mm):
    rain_mm = _to_float_or_nan(rain_mm)
    if np.isnan(rain_mm):
        return "gray"
    if rain_mm < 10:
        return "green"
    elif rain_mm < 40:
        return "yellow"
    elif rain_mm < 80:
        return "orange"
    return "red"


def dynamic_marker_color(level):
    mapping = {
        "SAFE": "green",
        "WARNING": "orange",
        "URGENT": "red",
        "CRITICAL": "darkred",
    }
    return mapping.get(str(level).upper(), "blue")


def classify_terrain_risk(row):
    """
    Match the W4 homework logic, but harmonize output labels to the
    assignment specification: HIGH / MEDIUM / LOW.

    W4 original logic:
    - very_high: river_distance < 500 and max_slope > threshold
    - high:      river_distance < 500 or  max_slope > threshold
    - medium:    river_distance < 1000 and mean_elevation < threshold
    - low:       otherwise

    Assignment-compliant mapping:
    - HIGH   <- very_high or high
    - MEDIUM <- medium
    - LOW    <- low
    """
    river_distance = row.get("river_distance_m", np.nan)
    mean_elev = row.get("mean_elevation", np.nan)
    max_slope = row.get("max_slope", np.nan)

    if pd.notna(river_distance) and pd.notna(max_slope) and river_distance < 500 and max_slope > SLOPE_THRESHOLD:
        return "HIGH"
    elif (pd.notna(river_distance) and river_distance < 500) or (pd.notna(max_slope) and max_slope > SLOPE_THRESHOLD):
        return "HIGH"
    elif pd.notna(river_distance) and pd.notna(mean_elev) and river_distance < 1000 and mean_elev < ELEVATION_LOW:
        return "MEDIUM"
    else:
        return "LOW"


def classify_dynamic_risk(row):
    """
    Dynamic override logic required by the assignment.

    CRITICAL: rain_1hr > 80 mm and inside the critical-rain buffer
    URGENT:   rain_1hr > 40 mm and terrain_risk == HIGH
    WARNING:  rain_1hr > 40 mm or terrain_risk == HIGH
    SAFE:     otherwise
    """
    terrain_high = str(row.get("terrain_risk", "")).upper() == "HIGH"

    if bool(row.get("is_critical_zone", False)):
        return "CRITICAL"
    if bool(row.get("is_warning_zone", False)) and terrain_high:
        return "URGENT"
    if bool(row.get("is_warning_zone", False)) or terrain_high:
        return "WARNING"
    return "SAFE"


## Captain's Log 3 — Load W3/W4 base layers

Load township boundaries, shelters, and river polygons.  
Then convert everything to **EPSG:3826** so distance and buffers use meters.

This notebook uses:
- township boundaries from the National Land Surveying and Mapping Center
- shelter CSV from the prior ARIA work
- WRA river polygons

In [17]:

# --- Load township polygons ---
assert township_path.exists(), f"Township shapefile not found: {township_path}"
townships = gpd.read_file(township_path).to_crs(epsg=3826)

# Hualien + Yilan station processing region
target_station_counties = ["花蓮縣", "宜蘭縣"]
station_region = townships[townships["COUNTYNAME"].isin(target_station_counties)].copy()
assert len(station_region) > 0, "Could not find Hualien + Yilan township polygons."

# Main county analysis area
county_towns = townships[townships["COUNTYNAME"] == TARGET_COUNTY].copy()
assert len(county_towns) > 0, f"No township polygons found for {TARGET_COUNTY}"

county_boundary = county_towns.dissolve().reset_index(drop=True)
station_region_boundary = station_region.dissolve().reset_index(drop=True)

# --- Load shelters CSV ---
assert shelter_csv_path.exists(), f"Shelter CSV not found: {shelter_csv_path}"
shelters_raw = pd.read_csv(shelter_csv_path, encoding="utf-8")

# Try to infer coordinate and name columns used in the official CSV
name_candidates = ["避難收容處所名稱", "名稱", "name"]
lat_candidates = ["緯度", "Latitude", "lat", "LAT"]
lon_candidates = ["經度", "Longitude", "lon", "LON"]

name_col = next((c for c in name_candidates if c in shelters_raw.columns), None)
lat_col = next((c for c in lat_candidates if c in shelters_raw.columns), None)
lon_col = next((c for c in lon_candidates if c in shelters_raw.columns), None)

assert lat_col is not None and lon_col is not None, "Cannot find shelter latitude/longitude columns."
assert name_col is not None, "Cannot find shelter name column."

shelters_raw[lat_col] = pd.to_numeric(shelters_raw[lat_col], errors="coerce")
shelters_raw[lon_col] = pd.to_numeric(shelters_raw[lon_col], errors="coerce")

# Basic coordinate cleanup
shelters_raw = shelters_raw[
    shelters_raw[lat_col].between(21, 26) &
    shelters_raw[lon_col].between(119, 123)
].copy()

shelters = gpd.GeoDataFrame(
    shelters_raw,
    geometry=gpd.points_from_xy(shelters_raw[lon_col], shelters_raw[lat_col]),
    crs="EPSG:4326"
).to_crs(epsg=3826)

# Keep shelters inside target county
shelters = gpd.sjoin(shelters, county_towns[["geometry"]], how="inner", predicate="within").drop(columns=["index_right"])
shelters = shelters.reset_index(drop=True)

# --- Download / load WRA river polygons ---
river_extract_dir.mkdir(parents=True, exist_ok=True)
river_zip_path = river_extract_dir / "riverpoly.zip"

if not any(river_extract_dir.glob("*.shp")):
    try:
        resp = requests.get(river_zip_url, timeout=60)
        resp.raise_for_status()
        with open(river_zip_path, "wb") as f:
            f.write(resp.content)
        import zipfile
        with zipfile.ZipFile(river_zip_path, "r") as zf:
            zf.extractall(river_extract_dir)
        print("River polygons downloaded and extracted.")
    except Exception as e:
        raise RuntimeError(f"Failed to download river polygons: {e}")

river_shps = list(river_extract_dir.glob("*.shp"))
assert river_shps, f"No river shapefile found in {river_extract_dir}"
rivers = gpd.read_file(river_shps[0]).to_crs(epsg=3826)

# Clip rivers to county buffer to keep only relevant geometry
county_buffer = county_boundary.copy()
county_buffer["geometry"] = county_buffer.buffer(5000)
rivers_county = gpd.overlay(rivers, county_buffer, how="intersection")

print("Townships:", len(townships))
print("County townships:", len(county_towns))
print("Shelters in county:", len(shelters))
print("River features (county area):", len(rivers_county))

Townships: 368
County townships: 13
Shelters in county: 198
River features (county area): 741


## Captain's Log 4 — W3 river-distance baseline

Compute the minimum shelter-to-river distance in meters.  
This preserves the W3 river exposure idea and becomes one component of the static risk layer.

In [18]:

shelters["river_distance_m"] = shelters.geometry.apply(lambda geom: rivers_county.distance(geom).min())

def river_distance_category(d):
    if d < 500:
        return "<500m"
    elif d < 1000:
        return "500-1000m"
    return ">=1000m"

shelters["river_distance_category"] = shelters["river_distance_m"].apply(river_distance_category)

river_summary = shelters[[name_col, "river_distance_m", "river_distance_category"]].head()
display(river_summary)

,避難收容處所名稱,river_distance_m,river_distance_category
0,和平國小,865.673162,500-1000m
1,豐南社區活動中心,53.524272,<500m
2,玉寶宮,277.175989,<500m
3,永豐社區活動中心,342.785916,<500m
4,富里老人文康中心,294.440576,<500m


## Captain's Log 5 — W4 terrain layer: DEM clip and slope calculation

Read the 20 m DEM, clip it to the county boundary, and derive the slope raster in degrees.  
Then extract terrain statistics for 500 m shelter buffers.

In [6]:

assert dem_path.exists(), f"DEM not found: {dem_path}"

dem = rxr.open_rasterio(dem_path, masked=True)

clip_boundary = county_boundary.copy()
if str(dem.rio.crs) != str(clip_boundary.crs):
    clip_boundary = clip_boundary.to_crs(dem.rio.crs)

dem_clip = dem.rio.clip(clip_boundary.geometry, clip_boundary.crs, drop=True)

dem_arr = dem_clip.values[0].astype("float64")
if np.ma.isMaskedArray(dem_arr):
    dem_arr = dem_arr.filled(np.nan)

res_x, res_y = dem_clip.rio.resolution()
pixel_size = abs(res_x)

dy, dx = np.gradient(dem_arr, pixel_size)
slope_arr = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

dem_affine = dem_clip.rio.transform()

print("DEM shape:", dem_clip.shape)
print("Pixel size:", pixel_size)
print("Elevation range:", float(np.nanmin(dem_arr)), "to", float(np.nanmax(dem_arr)))
print("Slope range:", float(np.nanmin(slope_arr)), "to", float(np.nanmax(slope_arr)))

DEM shape: (1, 7054, 3997)
Pixel size: 20.0
Elevation range: -3.0199999809265137 to 3824.300048828125
Slope range: 0.0 to 85.44615229262399


## Captain's Log 6 — W4 shelter zonal statistics and static terrain risk

Generate a 500 m shelter buffer, compute mean elevation / elevation spread / max slope, and classify the static terrain risk.

To stay consistent with the original W4 homework while matching the final assignment spec, the original four-rule logic is preserved but its labels are harmonized into exactly three values:

- **HIGH**: W4 `very_high` or `high`
- **MEDIUM**: W4 `medium`
- **LOW**: W4 `low`

This keeps the W4 decision rules intact while preventing a label mismatch in ARIA v3.0.


In [7]:

shelter_buffers = shelters.copy()
shelter_buffers["geometry"] = shelter_buffers.buffer(BUFFER_SHELTER_M)

zs_elev = zonal_stats(
    shelter_buffers,
    dem_arr,
    affine=dem_affine,
    stats=["mean", "std"],
    nodata=np.nan,
)

zs_slope = zonal_stats(
    shelter_buffers,
    slope_arr,
    affine=dem_affine,
    stats=["max"],
    nodata=np.nan,
)

terrain_df = pd.DataFrame(zs_elev).rename(columns={
    "mean": "mean_elevation",
    "std": "std_elevation",
})
terrain_df["max_slope"] = pd.DataFrame(zs_slope)["max"]

shelters = pd.concat([shelters.reset_index(drop=True), terrain_df], axis=1)
shelters["terrain_risk"] = shelters.apply(classify_terrain_risk, axis=1)

display(
    shelters[[name_col, "river_distance_m", "mean_elevation", "max_slope", "terrain_risk"]].head()
)

,避難收容處所名稱,river_distance_m,mean_elevation,max_slope,terrain_risk
0,和平國小,865.673162,27.502351,45.473255,HIGH
1,豐南社區活動中心,53.524272,299.459149,41.046371,HIGH
2,玉寶宮,277.175989,240.630908,29.891893,HIGH
3,永豐社區活動中心,342.785916,257.456815,32.286249,HIGH
4,富里老人文康中心,294.440576,238.324347,30.526334,HIGH


## Captain's Log 7 — Mode switcher, JSON normalization, fallback, and CRS handoff

Read `APP_MODE` from `.env`.

- `LIVE`: fetch the CWA API
- `SIMULATION`: load `fungwong_202511.json`
- if `LIVE` fails: automatically fall back to the latest local snapshot

Then perform the CRS handoff:

1. normalized rain stations start in **EPSG:4326**
2. analysis converts them to **EPSG:3826**
3. final Folium rendering converts them back to **EPSG:4326**


In [8]:

rain_source = None
raw_rain_json = None

if APP_MODE == "LIVE":
    raw_rain_json = fetch_cwa_api(CWA_API_KEY)
    if raw_rain_json is not None:
        rain_source = "LIVE"
    else:
        snapshot_path = get_latest_local_snapshot(SIMULATION_DATA)
        with open(snapshot_path, "r", encoding="utf-8") as f:
            raw_rain_json = json.load(f)
        rain_source = "FALLBACK_LOCAL_SNAPSHOT"
else:
    snapshot_path = get_latest_local_snapshot(SIMULATION_DATA)
    with open(snapshot_path, "r", encoding="utf-8") as f:
        raw_rain_json = json.load(f)
    rain_source = "SIMULATION"

# 1) normalize into WGS84 / EPSG:4326
rain_stations_4326 = normalize_cwa_json(raw_rain_json)
assert str(rain_stations_4326.crs) == "EPSG:4326", "Normalized rain stations must start in EPSG:4326"

# 2) convert to EPSG:3826 for all spatial analysis
rain_stations = rain_stations_4326.to_crs(epsg=3826)
assert str(rain_stations.crs) == "EPSG:3826", "Rain stations must be reprojected to EPSG:3826 for analysis"

# Keep full Hualien + Yilan station dataset for the assignment
rain_stations = gpd.sjoin(
    rain_stations,
    station_region[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"]).reset_index(drop=True)

print("Rain source:", rain_source)
print("Stations after Hualien + Yilan filter:", len(rain_stations))
display(rain_stations.head())


Rain source: SIMULATION
Stations after Hualien + Yilan filter: 170


,station_id,station_name,obs_time,lon,lat,rain_10min,rain_1hr,rain_3hr,rain_6hr,rain_12hr,rain_24hr,geometry
0,C0TA10,花蓮漁港,2025-11-11T18:50:00+08:00,121.638331,23.995322,0.0,0.0,1.0,7.5,77.5,130.0,POINT (314948.07 2654652.181)
1,C0TA20,加灣,2025-11-11T18:50:00+08:00,121.608944,24.082025,0.0,0.0,0.5,4.5,52.5,137.5,POINT (311916.388 2664241.469)
2,C0TA30,鹽寮,2025-11-11T18:50:00+08:00,121.592664,23.845606,0.0,0.5,0.5,1.5,65.0,136.5,POINT (310371.033 2638050.835)
3,C0UB60,明池,2025-11-11T18:50:00+08:00,121.475277,24.650688,1.5,6.0,17.5,55.5,107.5,265.0,POINT (298109.566 2727171.265)
4,C0UB70,太平山中間站,2025-11-11T18:50:00+08:00,121.513453,24.542003,0.5,3.5,10.0,31.5,54.5,102.0,POINT (302018.871 2715147.357)


## Captain's Log 8 — High-rainfall buffers and dynamic shelter impact join

Build 5 km buffers around rainfall stations and use `gpd.sjoin()` to find shelters inside the impact zones.

Required logic:
- **CRITICAL** zone: station `rain_1hr > 80 mm`
- **WARNING** zone: station `rain_1hr > 40 mm`

> Defensive check  
> We assert matching CRS before the joins, otherwise `sjoin()` can silently return empty results.

In [9]:

# Preserve original station points for later nearest-station matching and map display
warning_stations = rain_stations[rain_stations["rain_1hr"] > RAIN_WARNING_MM].copy()
critical_stations = rain_stations[rain_stations["rain_1hr"] > RAIN_CRITICAL_MM].copy()

warning_buffers = warning_stations.copy()
critical_buffers = critical_stations.copy()

warning_buffers["geometry"] = warning_buffers.buffer(BUFFER_RAIN_M)
critical_buffers["geometry"] = critical_buffers.buffer(BUFFER_RAIN_M)

assert str(shelters.crs) == str(rain_stations.crs), "CRS MISMATCH!"
assert str(shelters.crs) == str(warning_buffers.crs), "CRS MISMATCH!"
assert str(shelters.crs) == str(critical_buffers.crs), "CRS MISMATCH!"

if len(warning_buffers) > 0:
    shelters_warning = gpd.sjoin(
        shelters[["geometry"]],
        warning_buffers[["geometry"]],
        how="left",
        predicate="intersects"
    )
    shelters["is_warning_zone"] = shelters_warning["index_right"].notna().values
else:
    shelters["is_warning_zone"] = False

if len(critical_buffers) > 0:
    shelters_critical = gpd.sjoin(
        shelters[["geometry"]],
        critical_buffers[["geometry"]],
        how="left",
        predicate="intersects"
    )
    shelters["is_critical_zone"] = shelters_critical["index_right"].notna().values
else:
    shelters["is_critical_zone"] = False

print("Shelters in >40 mm/hr zone:", int(shelters["is_warning_zone"].sum()))
print("Shelters in >80 mm/hr zone:", int(shelters["is_critical_zone"].sum()))

Shelters in >40 mm/hr zone: 0
Shelters in >80 mm/hr zone: 0


## Captain's Log 9 — Nearest rain station attachment and dynamic risk grading

Attach the nearest rainfall station to each shelter so the popup can show:
- nearest station name
- nearest station hourly rainfall

Then run the required dynamic grading:

- **CRITICAL**: `rain_1hr > 80 mm` impact zone, regardless of original risk  
- **URGENT**: `rain_1hr > 40 mm` and `terrain_risk == HIGH`  
- **WARNING**: `rain_1hr > 40 mm` or `terrain_risk == HIGH`  
- **SAFE**: otherwise


In [10]:

nearest_cols = [
    "station_id", "station_name", "rain_1hr", "rain_3hr", "rain_24hr", "geometry"
]

shelters = gpd.sjoin_nearest(
    shelters,
    rain_stations[nearest_cols],
    how="left",
    distance_col="nearest_station_distance_m",
    lsuffix="shelter",
    rsuffix="station"
)

# Clean up nearest-station columns after sjoin_nearest
if "station_name_station" in shelters.columns:
    shelters["nearest_station_name"] = shelters["station_name_station"]
    shelters["nearest_rain_1hr"] = shelters["rain_1hr"]
    shelters["nearest_rain_3hr"] = shelters["rain_3hr"]
    shelters["nearest_rain_24hr"] = shelters["rain_24hr"]
else:
    shelters["nearest_station_name"] = shelters.get("station_name", np.nan)
    shelters["nearest_rain_1hr"] = shelters.get("rain_1hr", np.nan)
    shelters["nearest_rain_3hr"] = shelters.get("rain_3hr", np.nan)
    shelters["nearest_rain_24hr"] = shelters.get("rain_24hr", np.nan)

shelters["dynamic_risk"] = shelters.apply(classify_dynamic_risk, axis=1)

# Optional continuous score for ranking
shelters["river_risk"] = 1 - minmax_scale(shelters["river_distance_m"])
shelters["slope_risk"] = minmax_scale(shelters["max_slope"].fillna(shelters["max_slope"].median()))
shelters["rain_risk"] = minmax_scale(shelters["nearest_rain_1hr"].fillna(0))

shelters["risk_score"] = (
    0.35 * shelters["river_risk"] +
    0.30 * shelters["slope_risk"] +
    0.35 * shelters["rain_risk"]
)

display_cols = [
    name_col, "terrain_risk", "dynamic_risk",
    "nearest_station_name", "nearest_rain_1hr",
    "river_distance_m", "max_slope", "risk_score"
]
display(shelters[display_cols].sort_values("risk_score", ascending=False).head(10))

,避難收容處所名稱,terrain_risk,dynamic_risk,nearest_station_name,nearest_rain_1hr,river_distance_m,max_slope,risk_score
5,石平社區活動中心,HIGH,WARNING,富里,2.0,201.079479,49.298894,0.855040
122,文蘭活動中心,HIGH,WARNING,鯉魚潭,1.5,229.364729,53.100337,0.778484
119,池南活動中心,HIGH,WARNING,鯉魚潭,1.5,97.664743,45.408791,0.767241
2,玉寶宮,HIGH,WARNING,富里,2.0,277.175989,29.891893,0.766900
4,富里老人文康中心,HIGH,WARNING,富里,2.0,294.440576,30.526334,0.766890
3,永豐社區活動中心,HIGH,WARNING,富里,2.0,342.785916,32.286249,0.766797
107,西林天主堂,HIGH,WARNING,新東礦,1.0,537.266901,79.401113,0.750274
128,銅門收容所,HIGH,WARNING,銅門,1.0,190.714085,65.691610,0.746600
108,月眉國小,HIGH,WARNING,大坑,1.5,112.992518,33.331425,0.717078
109,月眉國小-米棧分校,HIGH,WARNING,大坑,1.5,125.837768,33.331425,0.715198


## Captain's Log 10 — Folium interactive map

Build the required interactive map:

1. **Basemap** centered on Hualien County
2. **Rainfall CircleMarker layer** with radius proportional to hourly rainfall and 4-level colors
3. **Shelter marker layer** colored by dynamic risk
4. **HeatMap layer** for rainfall intensity
5. **LayerControl**
6. **Popup** with shelter name, terrain risk, dynamic risk, nearest rain station, and hourly rainfall

Important Folium rule: coordinate order is always **[latitude, longitude]**.


In [11]:

# Map center from county boundary centroid (convert back to WGS84 for Folium)
county_boundary_wgs84 = county_boundary.to_crs(epsg=4326)
county_centroid_wgs84 = county_boundary_wgs84.geometry.iloc[0].centroid
map_center = [county_centroid_wgs84.y, county_centroid_wgs84.x]

m = folium.Map(
    location=map_center,
    zoom_start=10,
    tiles="CartoDB positron"
)

# Feature groups
fg_rain = FeatureGroup(name="Rainfall Stations", show=True)
fg_shelter = FeatureGroup(name="Shelters (Dynamic Risk)", show=True)
fg_heat = FeatureGroup(name="Rainfall HeatMap", show=False)

# Convert to WGS84 for Folium display
rain_wgs84 = rain_stations.to_crs(epsg=4326).copy()
shelters_wgs84 = shelters.to_crs(epsg=4326).copy()

assert str(rain_wgs84.crs) == "EPSG:4326", "Folium rain layer must be EPSG:4326"
assert str(shelters_wgs84.crs) == "EPSG:4326", "Folium shelter layer must be EPSG:4326"

# Rain CircleMarkers
for _, row in rain_wgs84.iterrows():
    rain_1hr = _to_float_or_nan(row.get("rain_1hr"))
    radius = max(5, 0 if np.isnan(rain_1hr) else rain_1hr / 5.0)
    popup_html = f"""
    <b>Rain Station</b><br>
    Name: {row.get('station_name', '')}<br>
    ID: {row.get('station_id', '')}<br>
    1hr Rain: {rain_1hr:.1f} mm<br>
    3hr Rain: {_to_float_or_nan(row.get('rain_3hr')):.1f} mm<br>
    24hr Rain: {_to_float_or_nan(row.get('rain_24hr')):.1f} mm
    """
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],   # Folium requires [latitude, longitude]
        radius=radius,
        color=rain_color(rain_1hr),
        fill=True,
        fill_color=rain_color(rain_1hr),
        fill_opacity=0.75,
        weight=1,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row.get('station_name', '')}: {rain_1hr:.1f} mm/hr"
    ).add_to(fg_rain)

# HeatMap data
heat_data = []
for _, row in rain_wgs84.iterrows():
    rain_1hr = _to_float_or_nan(row.get("rain_1hr"))
    if not np.isnan(rain_1hr):
        heat_data.append([row.geometry.y, row.geometry.x, rain_1hr])  # lat, lon, weight

if heat_data:
    HeatMap(heat_data, name="Rainfall HeatMap", radius=20, blur=18, min_opacity=0.2).add_to(fg_heat)

# Shelter markers
for _, row in shelters_wgs84.iterrows():
    popup_html = f"""
    <b>{row.get(name_col, 'Shelter')}</b><br>
    Terrain Risk: {row.get('terrain_risk', '')}<br>
    Dynamic Risk: {row.get('dynamic_risk', '')}<br>
    Nearest Rain Station: {row.get('nearest_station_name', '')}<br>
    Nearest 1hr Rain: {_to_float_or_nan(row.get('nearest_rain_1hr')):.1f} mm<br>
    River Distance: {_to_float_or_nan(row.get('river_distance_m')):.1f} m<br>
    Max Slope: {_to_float_or_nan(row.get('max_slope')):.1f}°
    """
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],   # Folium requires [latitude, longitude]
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{row.get(name_col, 'Shelter')} | {row.get('dynamic_risk', '')}",
        icon=folium.Icon(color=dynamic_marker_color(row.get("dynamic_risk")), icon="home", prefix="fa")
    ).add_to(fg_shelter)

fg_rain.add_to(m)
fg_shelter.add_to(m)
fg_heat.add_to(m)
LayerControl(collapsed=False).add_to(m)

m.save(str(html_output_path))
print(f"Saved interactive map to: {html_output_path}")

m


Saved interactive map to: C:\Users\admin\Desktop\遙測\hw5\output\ARIA_v3_Fungwong.html


## Captain's Log 11 — Persist outputs

Save the shelter result table for reporting and verification.

In [12]:

# Keep only one active geometry column for export
export_gdf = shelters.copy()

export_gdf.to_file(geojson_output_path, driver="GeoJSON")
export_gdf.drop(columns="geometry").to_csv(csv_output_path, index=False, encoding="utf-8-sig")

print(f"CSV saved: {csv_output_path}")
print(f"GeoJSON saved: {geojson_output_path}")
print(f"HTML saved: {html_output_path}")

CSV saved: C:\Users\admin\Desktop\遙測\hw5\output\ARIA_v3_shelters.csv
GeoJSON saved: C:\Users\admin\Desktop\遙測\hw5\output\ARIA_v3_shelters.geojson
HTML saved: C:\Users\admin\Desktop\遙測\hw5\output\ARIA_v3_Fungwong.html


## Captain's Log 12 — Quick QA checks

Run a few checks that are likely to catch the most common homework failures early.

In [13]:

print("Three-way CRS checks:")
print("  rain_stations_4326:", rain_stations_4326.crs)
print("  rain_stations analysis:", rain_stations.crs)
print("  shelters analysis:", shelters.crs)

assert str(rain_stations_4326.crs) == "EPSG:4326", "Raw/normalized rain stations should be EPSG:4326"
assert str(rain_stations.crs) == "EPSG:3826", "Analysis rain stations should be EPSG:3826"
assert str(shelters.crs) == "EPSG:3826", "Shelters should be EPSG:3826"

print("\nRainfall no-data check:")
print("  Remaining NaN 1hr rain rows:", int(rain_stations["rain_1hr"].isna().sum()))
assert int(rain_stations["rain_1hr"].isna().sum()) == 0, "All -998 / missing 1hr rain should be filtered before GDF creation"

print("\nDynamic risk counts:")
display(shelters["dynamic_risk"].value_counts(dropna=False))

print("\nTerrain risk counts:")
display(shelters["terrain_risk"].value_counts(dropna=False))

print("\nTop shelters by risk score:")
display(
    shelters[[name_col, "dynamic_risk", "terrain_risk", "nearest_station_name", "nearest_rain_1hr", "risk_score"]]
    .sort_values("risk_score", ascending=False)
    .head(10)
)


Three-way CRS checks:
  rain_stations_4326: EPSG:4326
  rain_stations analysis: EPSG:3826
  shelters analysis: EPSG:3826

Rainfall no-data check:
  Remaining NaN 1hr rain rows: 0

Dynamic risk counts:


dynamic_risk
WARNING    124
SAFE        74
Name: count, dtype: int64


Terrain risk counts:


terrain_risk
HIGH      124
LOW        52
MEDIUM     22
Name: count, dtype: int64


Top shelters by risk score:


,避難收容處所名稱,dynamic_risk,terrain_risk,nearest_station_name,nearest_rain_1hr,risk_score
5,石平社區活動中心,WARNING,HIGH,富里,2.0,0.855040
122,文蘭活動中心,WARNING,HIGH,鯉魚潭,1.5,0.778484
119,池南活動中心,WARNING,HIGH,鯉魚潭,1.5,0.767241
2,玉寶宮,WARNING,HIGH,富里,2.0,0.766900
4,富里老人文康中心,WARNING,HIGH,富里,2.0,0.766890
3,永豐社區活動中心,WARNING,HIGH,富里,2.0,0.766797
107,西林天主堂,WARNING,HIGH,新東礦,1.0,0.750274
128,銅門收容所,WARNING,HIGH,銅門,1.0,0.746600
108,月眉國小,WARNING,HIGH,大坑,1.5,0.717078
109,月眉國小-米棧分校,WARNING,HIGH,大坑,1.5,0.715198


## Appendix — README AI diagnostic log template

Copy this section into your `README.md` and expand at least one item.

### AI Diagnostic Log

1. **Lat/Lon reversed in Folium**  
   - Symptom: markers appear in the ocean or outside Taiwan.  
   - Fix: Folium always expects `[latitude, longitude]`, not `[longitude, latitude]`.

2. **CWA API returned `-998`**  
   - Symptom: impossible colors, giant circles, or broken rain classification.  
   - Fix: convert `-998` to `NaN` and filter rows **before** creating the rainfall GeoDataFrame.

3. **`gpd.sjoin()` returned empty results**  
   - Symptom: no shelters fall inside rain buffers even during extreme rainfall.  
   - Fix: verify CRS alignment first. Rain stations must be projected to **EPSG:3826** before buffering and spatial join.

4. **HeatMap blind spots in mountain areas**  
   - Symptom: highland rainfall intensity looks underrepresented.  
   - Fix: explain that station coverage is uneven, so the HeatMap visualizes sampled observations rather than a continuous rainfall surface.

5. **Live API coordinate trap**  
   - Symptom: stations shift by about 1 km relative to expected positions.  
   - Fix: in LIVE mode, `GeoInfo.Coordinates[0]` is TWD67 and `[1]` is WGS84. The parser must explicitly use the WGS84 coordinate set.


Path portability note: this final version removes notebook hard-coded data paths and resolves files through `.env`, relative paths, and discovery patterns.
